# Radio spectra in PySPEDAS

Short presentation for the "Bridging gaps in heliospheric radio data analyses" workshop
2026 May 18-22

From the PySPEDAS documentation (https://pyspedas.readthedocs.io/)

>
> PySPEDAS is an implementation of the Space Physics Environment Data Analysis Software (SPEDAS) framework in Python.
>
> The SPEDAS framework is written in IDL and contains data loading, data analysis and data plotting tools for various scientific missions (NASA, NOAA, etc.) and ground magnetometers.
>

An example of using PySPEDAS (together with sunpy):

https://github.com/spedas/pyspedas_examples/blob/master/pyspedas_examples/notebooks/Exploring_the_Heliosphere_with_Python.ipynb

This notebook loads PSP/RFS, Wind/WAVES, STEREO/WAVES, and Solar Orbiter/RPW radio spectra. By default the PySPEDAS routines for these spacecraft download data from CDAWeb.

## Getting Started

### Data Levels

The heliospheric radio spacecraft shown as examples in this notebook produce Level 2 and Level 3 data files in CDF format. Level 2 files typically include data in units of power spectral density (PSD, $\mathrm{V^2/Hz}$), and Level 3 files include calibrated data in units of flux ($\mathrm{W/m^2/Hz}$ or $\mathrm{sfu}$).

In particular, the Level 3 data files shown here include variables with names ending in `PSD_FLUX` or `PSD_SFU`. The `FLUX` variables are in units of calibrated flux, $\mathrm{W/m^2/Hz}$, accounting for spacecraft antenna effective length and gain, but not scaling based on radial distance from the Sun. The `SFU` variables are scaled by $(r/r_{au})^2$. This normalization can be useful for multi-spacecraft investigation of angular dependence of solar radio emission.

### Use cases

Many users will be interested primarily in plotting a radio dynamic spectrum (for example, a user interested in finding the start time and drift rate of a Type III radio burst). For this use case, we suggest using Level 3   `PSD_FLUX` or `PSD_SFU` data, depending on whether the user is interested in absolute flux or flux normalized to 1 au.

For some use cases, a user should not use these defaults?

  - For tasks which are dependent on the actual antenna used in the observation, such as direction finding/goniopolarimetry, the data from each antenna will be useful.
  - The spacecraft data here include both radio (remote emission) and plasma (in situ electric field) signals. For investigation of in situ signals (e.g. quasi-thermal noise analysis, plasma waves) units of $\mathrm{V^2/Hz}$ are more useful than the flux units.

## Setup

Package imports: note that `pytplot` used to be a separate package, but is now built in to PySPEDAS for versions >2.0

In [ ]:
import pyspedas.projects.psp as psp
import pyspedas.projects.wind as wind
import pyspedas.projects.stereo as stereo
import pyspedas.projects.solo as solo

from pyspedas import tnames, tplot, options, get_data, store_data

Set a time range:

In [ ]:
trange = ['2024-03-23/00:00:00', '2024-03-23/04:00:00']

## PSP/FIELDS RFS

Load RFS LFR and HFR data files. 

By default PySPEDAS checks for updates on CDAWeb, this can be disabled with the `no_update` keyword.

The call below loads the L3 data (L2 data is loaded by default).

The line below downloads the CDF and loads the CDF variables into TPLOT variables. The `tnames` function displays all the RFS variables currently loaded into the notebook session.

In [ ]:
psp.fields(trange=trange, time_clip=True, datatype='rfs_hfr', level = 'l3')
psp.fields(trange=trange, time_clip=True, datatype='rfs_lfr', level = 'l3')

tnames('*rfs*')

The basic command to plot spectrogram data is `tplot`.

In [ ]:
tplot(['psp_fld_l3_rfs_hfr_PSD_FLUX','psp_fld_l3_rfs_lfr_PSD_FLUX'])

## Wind/WAVES

Load and plot Wind/WAVES RAD2, RAD1, and TNR data from CDAWeb.

In [ ]:
wind.waves(trange=trange, time_clip=True, datatype='rad2')
wind.waves(trange=trange, time_clip=True, datatype='rad1')
wind.waves(trange=trange, time_clip=True, datatype='tnr')

tplot(['wi_l2_wav_rad?_PSD_V2_Z','wi_l2_wav_tnr_PSD_V2'])

## STEREO/WAVES

Load and plot STEREO Ahead/WAVES HFR and LFR data from CDAWeb (using `probe = 'a'` to select Ahead). 

By default, the TPLOT variable name is simply the CDF variable name. Some CDF variables include the spacecraft name and some don't. For those that don't, we can add a `prefix` to distinguish spacecraft and receiver in the TPLOT variable name.

In [ ]:
stereo.waves(trange=trange, time_clip=True, datatype='hfr', probe = 'a', prefix = 'sta_swaves_hfr_')
stereo.waves(trange=trange, time_clip=True, datatype='lfr', probe = 'a', prefix = 'sta_swaves_lfr_')

tplot('sta_swaves_?fr_PSD_FLUX')

## Solar Orbiter/RPW

Load and plot Solar Orbiter/RPW HFR and TNR data from CDAWeb

In [ ]:
solo.rpw(trange=trange, time_clip=True, datatype='hfr-surv-flux', level = 'l3', prefix = 'solo_l3_rpw_hfr-surv-flux_')
solo.rpw(trange=trange, time_clip=True, datatype='tnr-surv-flux', level = 'l3', prefix = 'solo_l3_rpw_tnr-surv-flux_')

In [ ]:
options('solo_l3_rpw_hfr-surv-flux_PSD_SFU', 'ylog',1)
options('solo_l3_rpw_hfr-surv-flux_PSD_SFU', 'zlog',1)
options('solo_l3_rpw_tnr-surv-flux_PSD_SFU', 'ylog',1)
options('solo_l3_rpw_tnr-surv-flux_PSD_SFU', 'zlog',1)
tplot(['solo_l3_rpw_hfr-surv-flux_PSD_SFU', 'solo_l3_rpw_tnr-surv-flux_PSD_SFU'])

## Plot options

Most options available in `matplotlib` can be set for TPLOT variables using the `options` function. Here we just do a couple simple settings to add spacecraft information to the Y title and make sure the receivers on each spacecraft use the same Z scale.

Plot options are stored in a structure along with the data.

In [ ]:
options('psp_fld_l3_rfs_hfr_PSD_SFU', 'ytitle', 'PSP/HFR')
options('psp_fld_l3_rfs_lfr_PSD_SFU', 'ytitle', 'PSP/LFR')
options('psp_fld_l3_rfs_?fr_PSD_SFU', 'zrange', [5e1,1e5])

options('wi_l2_wav_rad1_PSD_V2_Z', 'ytitle', 'Wind/RAD1')
options('wi_l2_wav_rad2_PSD_V2_Z', 'ytitle', 'Wind/RAD2')
options('wi_l2_wav_tnr_PSD_V2', 'ytitle', 'Wind/TNR')
options('wi_l2_wav*PSD_V2*', 'zrange', [1e-16,1e-12])

options('sta_swaves_hfr_PSD_SFU', 'ytitle', 'STA/HFR')
options('sta_swaves_lfr_PSD_SFU', 'ytitle', 'STA/LFR')
options('sta_swaves_?fr_PSD_SFU', 'zrange', [5e1,5e6])

options('solo_l3_rpw_hfr-surv-flux_PSD_SFU', 'ytitle', 'SolO/HFR')
options('solo_l3_rpw_tnr-surv-flux_PSD_SFU', 'ytitle', 'SolO/TNR')
options('solo_l3_rpw_???-surv-flux_PSD_SFU', 'zrange', [5e1,5e6])

In [ ]:
tplot(['psp_fld_l3_rfs_hfr_PSD_SFU','psp_fld_l3_rfs_lfr_PSD_SFU',
       'wi_l2_wav_rad?_PSD_V2_Z','wi_l2_wav_tnr_PSD_V2',
       'sta_swaves_?fr_PSD_SFU',
       'solo_l3_rpw_hfr-surv-flux_PSD_SFU',
       'solo_l3_rpw_tnr-surv-flux_PSD_SFU'], xsize = 10, ysize = 12)

## Data access

Data can be accessed from TPLOT variables using `get_data`. If `get_data` returns a variable `dat`, then 
  - Times are stored (Unix time) in `dat.times`
  - Spectral data is stored in the 2D array `dat.y`
  - Frequency data is stored in the 2D array `dat.v`

In [ ]:
rfs_hfr_data = get_data('psp_fld_l3_rfs_hfr_auto_averages_ch0_V1V2')

rfs_hfr_data

## Metadata access

The spacecraft CDF files obtained by this notebook follow the ISTP CDF metadata standard. The metadata and 

In [ ]:
rfs_hfr_meta = get_data('psp_fld_l3_rfs_hfr_PSD_SFU', metadata=True)

rfs_hfr_meta

Some useful metadata to examine are the TEXT in the CDF global attributes, the category description (CATDESC) in the variable attributes, and the variable notes (VAR_NOTES). VAR_NOTES is optional and not present for all variables.

The DOI global attribute (not yet present for all files) also points to a landing page which describes the contents of the CDF based on the metadata.

In [ ]:
rfs_hfr_meta['CDF']['GATT']['TEXT']

In [ ]:
rfs_hfr_meta['CDF']['VATT']['CATDESC']

In [ ]:
rfs_hfr_meta['CDF']['VATT']['VAR_NOTES']

In [ ]:
rfs_hfr_meta['CDF']['GATT']['DOI']

## Non-tplot examples

For users who prefer to use other tools for analysis and visualization, it is possible to use PySPEDAS without using the `tplot` plotting interface. This section illustrates two use cases:
  - `notplot`: downloading the file, and storing a CDF variable as a pandas DataFrame
  - `download_only`: downloading a CDF file without opening or loading it

### Storing data in a DataFrame

Setting `notplot=True` will download the requested CDF file, and loads it into a data structure. The keys of the data structure will correspond to the variables in the CDF file.

In [ ]:
rfs_hfr_struct = psp.fields(trange=trange, datatype='rfs_hfr', level='l3', notplot=True)

rfs_hfr_struct

We can store the data for a single CDF variable in a Pandas DataFrame, using the time (`x`) in the data structure as the DataFrame `index`, the data (`y`) from the structure as the DataFrame `data`, and the frequency (`v`) as the columns.

In [ ]:
import pandas as pd

rfs_var = rfs_hfr_struct['psp_fld_l3_rfs_hfr_PSD_SFU']
rfs_df = pd.DataFrame(index = rfs_var['x'], 
             data = rfs_var['y'],
             columns = rfs_var['v'][0,:])

rfs_df

Using the DataFrame we can plot a light curve at 1.77 MHz.

In [ ]:
rfs_df[1771875.0].plot(logy=True)

### Download only mode

PySPEDAS can also be used to simply download the requested CDF file. The file can then be read by a wide variety of CDF tools, including:
  - SciQLop [insert link to example notebook]
  - maser4py [insert link to example notebook]
  - SunPy radiospectra [insert link to example notebook]

The command below downloads the RFS file (if earlier sections of this notebook have already been run, it will confirm that the file is current and not re-download), and returns the full path of the file on the local drive.

In [ ]:
rfs_hfr_file = psp.fields(trange=trange, datatype='rfs_hfr', level='l3', downloadonly=True)

rfs_hfr_file